# Dataset Augmentation with Synthetic Waste Records

This notebook generates realistic synthetic canteen food-waste records (Jan 2015 – Dec 2024). Key realism improvements over the original version:

| Issue in original | Fix in this version |
|---|---|
| Waste uniform within meal window | Gaussian peak curve per meal (7:30 breakfast, 12:00 lunch, 18:00 dinner) |
| Background fill ≈ 2 kg per slot | Exponential(scale=0.05 kg) — near-zero off-peak signal |
| Weekend/event multipliers → foot traffic only | `waste_mult` applied directly to generated weight |
| No holidays | Full US public holiday calendar (2015-2025) → near-closure conditions |
| No seasonal calendar | Academic semester phases (summer break, spring break, winter break) |
| All sections identical | Per-section meal specialisation (section A = breakfast hub, D = dinner hub) |

All changes are backward-compatible: the output CSV has the same columns as before plus `Is_Holiday` and `Day_Type`.


## 1. Setup and Data Loading


In [ ]:
import os
import calendar
import pandas as pd
import numpy as np
from datetime import timedelta, datetime


In [ ]:
np.random.seed(42)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

try:
    os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
    print("Directory changed")
except OSError:
    print("Error: Can't change the Current Working Directory")


Mounted at /content/drive
Directory changed


In [ ]:
df_original = pd.read_csv('data/food_waste_cleaned.csv')
print(f"Original data shape: {df_original.shape}")


Original data shape: (2600, 23)


## 2. Add Realistic Times to Original Records
The original dataset only contains dates. We assign a 30-min-aligned timestamp within the correct meal window.


In [ ]:
# Meal window definitions (start hour inclusive, end hour exclusive)
meal_time_windows = {
    'Breakfast': (6, 9),
    'Lunch':     (11, 14),
    'Dinner':    (17, 20),
}

def add_realistic_times(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    def assign_time(row):
        start, end = meal_time_windows[row['Meal']]
        hour   = np.random.randint(start, end)
        minute = np.random.choice([0, 30])
        return row['Date'] + timedelta(hours=int(hour), minutes=int(minute))
    df['Date'] = df.apply(assign_time, axis=1)
    return df

df_original = add_realistic_times(df_original)
print('First few timestamps:')
print(df_original['Date'].head())


First few timestamps:
0   2025-06-11 08:30:00
1   2025-06-11 11:00:00
2   2025-06-11 19:30:00
3   2025-06-11 11:00:00
4   2025-06-11 19:30:00
Name: Date, dtype: datetime64[ns]


## 3. Analyse Original Data


In [ ]:
meal_dist            = df_original['Meal'].value_counts(normalize=True)
section_dist         = df_original['Canteen_Section'].value_counts(normalize=True)
category_dist        = df_original['Food_Category'].value_counts(normalize=True)
category_given_meal  = (
    df_original.groupby('Meal')['Food_Category']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)
print('Meal distribution:\n', meal_dist)
print('\nCanteen Section distribution:\n', section_dist)
print('\nFood Category distribution:\n', category_dist)
print('\nFood Category given Meal:\n', category_given_meal)


Meal distribution:
 Meal
Breakfast    0.334231
Lunch        0.333462
Dinner       0.332308
Name: proportion, dtype: float64

Canteen Section distribution:
 Canteen_Section
B    0.251538
A    0.250769
D    0.249231
C    0.248462
Name: proportion, dtype: float64

Food Category distribution:
 Food_Category
Rice          0.253462
Meat          0.251154
Vegetables    0.250385
Soup          0.245000
Name: proportion, dtype: float64

Food Category given Meal:
 Food_Category      Meat      Rice      Soup  Vegetables
Meal                                                   
Breakfast      0.245109  0.249712  0.255466    0.249712
Dinner         0.255787  0.261574  0.237269    0.245370
Lunch          0.252595  0.249135  0.242215    0.256055


In [ ]:
price_map = {'Meat': 8.0, 'Vegetables': 3.0, 'Soup': 1.5, 'Rice': 2.0}

df_original['Log_Waste'] = np.log(df_original['Waste_Weight_kg'])
category_params = {}
for cat in df_original['Food_Category'].unique():
    log_vals = df_original[df_original['Food_Category'] == cat]['Log_Waste']
    mu, sigma = log_vals.mean(), log_vals.std()
    category_params[cat] = {'mean_log': mu, 'std_log': sigma}
    print(f"{cat}: log_mean={mu:.4f}, log_std={sigma:.4f} => median={np.exp(mu):.2f} kg")

print('\nOriginal Waste_Weight_kg stats:')
print(df_original['Waste_Weight_kg'].describe())


Vegetables: log_mean=0.7206, log_std=0.8024 => median=2.06 kg
Soup: log_mean=0.7561, log_std=0.7875 => median=2.13 kg
Meat: log_mean=0.6427, log_std=0.8514 => median=1.90 kg
Rice: log_mean=0.7299, log_std=0.7933 => median=2.07 kg

Original Waste_Weight_kg stats:
count    2600.000000
mean        2.585988
std         1.408281
min         0.100000
25%         1.390000
50%         2.600000
75%         3.800000
max         5.000000
Name: Waste_Weight_kg, dtype: float64


In [ ]:
df_original['DayOfWeek'] = df_original['Date'].dt.dayofweek
events_per_meal_day = df_original.groupby(['Date', 'Meal']).size().reset_index(name='count')
events_per_meal_day['DayOfWeek'] = events_per_meal_day['Date'].dt.dayofweek
baseline_events = (
    events_per_meal_day.groupby(['Meal', 'DayOfWeek'])['count']
    .mean().round().astype(int).clip(lower=1)
)
print('Baseline events per meal per day:\n', baseline_events)


Baseline events per meal per day:
 Meal       DayOfWeek
Breakfast  0            3
           1            3
           2            3
           3            3
           4            3
           5            3
           6            3
Dinner     0            3
           1            3
           2            3
           3            3
           4            3
           5            3
           6            3
Lunch      0            3
           1            3
           2            2
           3            3
           4            3
           5            3
           6            3
Name: count, dtype: int64


## 4. Define Global Thresholds


In [ ]:
def compute_thresholds(df):
    q1_w = df['Waste_Weight_kg'].quantile(0.25)
    q3_w = df['Waste_Weight_kg'].quantile(0.75)
    iqr_w = q3_w - q1_w
    q1_c = df['Cost_Loss'].quantile(0.25)
    q3_c = df['Cost_Loss'].quantile(0.75)
    iqr_c = q3_c - q1_c
    return {
        'waste_lower': q1_w - 1.5*iqr_w, 'waste_upper': q3_w + 1.5*iqr_w,
        'cost_lower':  q1_c - 1.5*iqr_c, 'cost_upper':  q3_c + 1.5*iqr_c,
        'high_waste':  df['Waste_Weight_kg'].quantile(0.90),
        'high_cost':   df['Cost_Loss'].quantile(0.90),
    }

thresholds = compute_thresholds(df_original)
print(f"Waste outlier bounds: [{thresholds['waste_lower']:.2f}, {thresholds['waste_upper']:.2f}]")
print(f"High waste threshold (90th pct): {thresholds['high_waste']:.2f}")


Waste outlier bounds: [-2.23, 7.42]
High waste threshold (90th pct): 4.52


## 5. Helper Functions

### Overview of design decisions
1. **Intra-meal Gaussian peak** — slot-sampling probability and waste weight both decay as a Gaussian away from each meal's peak hour.
2. **`get_day_context()`** — single function combining weekday/weekend, event, holiday, and academic-calendar signals into `waste_mult` and `traffic_mult`.
3. **Waste weight = log-normal x waste_mult** — no separate meal/section multipliers that compound the log-normal scale.
4. **Background fill** uses `Exponential(scale=0.05 kg)` — near-zero, clearly distinguishable from event records in any time-series model.
5. **Section meal preferences** — each section has per-meal weights applied to event-count lambda only (not waste magnitude).


In [ ]:
# ── Intra-meal Gaussian peak curve ──────────────────────────────────────────
# Waste and footfall peak at a specific time within each meal window.
# The Gaussian decay reflects the real-world ramp-up / ramp-down pattern.

MEAL_PEAK_HOUR = {
    'Breakfast': 7.5,   # 7:30 AM
    'Lunch':    12.0,   # 12:00 PM
    'Dinner':   18.0,   # 6:00 PM
}
MEAL_PEAK_SIGMA = {
    'Breakfast': 0.75,
    'Lunch':     0.90,
    'Dinner':    1.00,
}


def get_intra_meal_multiplier(hour, minute, meal):
    """
    Return a weight in [0.25, 1.0] for the expected waste intensity at
    (hour, minute).  Gaussian centred on MEAL_PEAK_HOUR; clamped at 0.25
    so the quietest slot is not zero.
    """
    t     = hour + minute / 60.0
    mu    = MEAL_PEAK_HOUR[meal]
    sigma = MEAL_PEAK_SIGMA[meal]
    return max(np.exp(-0.5 * ((t - mu) / sigma) ** 2), 0.25)


def meal_30min_slots(meal):
    """Return list of (hour, minute) pairs covering the meal window."""
    start, end = meal_time_windows[meal]
    slots = []
    h = start
    while h < end:
        slots.extend([(h, 0), (h, 30)])
        h += 1
    return slots


def weighted_slot_sample(meal):
    """
    Sample a 30-min slot from the meal window with probability
    proportional to the Gaussian peak.  Peak slots are sampled more often,
    matching the real rush-hour pattern.
    """
    slots   = meal_30min_slots(meal)
    weights = np.array([get_intra_meal_multiplier(h, m, meal) for h, m in slots])
    weights /= weights.sum()
    return slots[np.random.choice(len(slots), p=weights)]


# Verify
print('Intra-meal slot weights (higher = more likely to be sampled):')
for meal in meal_time_windows:
    slots   = meal_30min_slots(meal)
    weights = [get_intra_meal_multiplier(h, m, meal) for h, m in slots]
    print(f'\n  {meal} (peak={MEAL_PEAK_HOUR[meal]:.1f}h):')
    for (h, m), w in zip(slots, weights):
        bar = chr(9608) * max(1, int(w * 20))
        print(f'    {h:02d}:{m:02d}  {w:.3f}  {bar}')


Intra-meal slot weights (higher = more likely to be sampled):

  Breakfast (peak=7.5h):
    06:00  0.250  █████
    06:30  0.411  ████████
    07:00  0.801  ████████████████
    07:30  1.000  ████████████████████
    08:00  0.801  ████████████████
    08:30  0.411  ████████

  Lunch (peak=12.0h):
    11:00  0.539  ██████████
    11:30  0.857  █████████████████
    12:00  1.000  ████████████████████
    12:30  0.857  █████████████████
    13:00  0.539  ██████████
    13:30  0.250  █████

  Dinner (peak=18.0h):
    17:00  0.607  ████████████
    17:30  0.882  █████████████████
    18:00  1.000  ████████████████████
    18:30  0.882  █████████████████
    19:00  0.607  ████████████
    19:30  0.325  ██████


In [ ]:
# ── Per-section meal specialisation ─────────────────────────────────────────
# Each section has a different throughput profile across meals.
# These weights scale the event-count lambda only, not waste magnitude.

SECTION_MEAL_PREF = {
    'A': {'Breakfast': 1.35, 'Lunch': 0.90, 'Dinner': 0.75},  # breakfast hub
    'B': {'Breakfast': 0.85, 'Lunch': 1.40, 'Dinner': 0.90},  # lunch hub
    'C': {'Breakfast': 1.00, 'Lunch': 1.05, 'Dinner': 0.95},  # balanced
    'D': {'Breakfast': 0.80, 'Lunch': 0.85, 'Dinner': 1.40},  # dinner hub
}
base_traffic_per_section = {'A': 100, 'B': 120, 'C': 80, 'D': 150}

print('Section meal preference weights:')
for sec, prefs in SECTION_MEAL_PREF.items():
    print(f'  Section {sec}: {prefs}')


Section meal preference weights:
  Section A: {'Breakfast': 1.35, 'Lunch': 0.9, 'Dinner': 0.75}
  Section B: {'Breakfast': 0.85, 'Lunch': 1.4, 'Dinner': 0.9}
  Section C: {'Breakfast': 1.0, 'Lunch': 1.05, 'Dinner': 0.95}
  Section D: {'Breakfast': 0.8, 'Lunch': 0.85, 'Dinner': 1.4}


In [ ]:
# ── US public holiday calendar ───────────────────────────────────────────────

def nth_weekday_of_month(year, month, weekday, n):
    """Return date of the n-th occurrence of weekday in month/year. n=-1 = last."""
    days = [d for d in calendar.monthcalendar(year, month) if d[weekday] != 0]
    day  = days[-1][weekday] if n == -1 else days[n-1][weekday]
    return pd.Timestamp(year=year, month=month, day=day)


def get_us_holidays(year):
    """Return a set of Timestamps for major US public holidays in year."""
    h = set()
    for month, day in [(1,1),(7,4),(11,11),(12,24),(12,25),(12,31)]:
        h.add(pd.Timestamp(year=year, month=month, day=day))
    h.add(nth_weekday_of_month(year,  1, 0,  3))   # MLK Day
    h.add(nth_weekday_of_month(year,  2, 0,  3))   # Presidents Day
    h.add(nth_weekday_of_month(year,  5, 0, -1))   # Memorial Day
    h.add(nth_weekday_of_month(year,  9, 0,  1))   # Labor Day
    thanksgiving = nth_weekday_of_month(year, 11, 3, 4)
    h.add(thanksgiving)
    h.add(thanksgiving + pd.Timedelta(days=1))     # Black Friday
    return h


def build_holiday_set(start_year, end_year):
    all_h = set()
    for yr in range(start_year, end_year + 1):
        all_h.update(get_us_holidays(yr))
    return all_h


# ── Academic semester calendar ────────────────────────────────────────────────

def get_academic_multiplier(date):
    """
    Return a base activity multiplier reflecting the campus population on site.

    Phase               Dates                Multiplier
    ──────────────────────────────────────────────────
    Winter break        Dec 16 - Jan 14      0.15
    Spring break        Mar 9-17             0.30
    Spring semester     Jan 15 - May 10      1.00
    Late-May transition May 11-14            0.70
    Summer session 1    May 15 - Jun 30      0.55
    Summer session 2    Jul 1 - Aug 20       0.45
    Orientation week    Aug 21-24            0.60
    Fall semester       Aug 25 - Dec 15      1.00
    """
    m, d = date.month, date.day
    if (m == 12 and d >= 16) or (m == 1 and d <= 14):  return 0.15
    if m == 3 and 9 <= d <= 17:                         return 0.30
    if (m == 1 and d >= 15) or (2 <= m <= 4) or (m == 5 and d <= 10):  return 1.00
    if m == 5 and 11 <= d <= 14:                        return 0.70
    if (m == 5 and d >= 15) or m == 6:                  return 0.55
    if m == 7 or (m == 8 and d <= 20):                  return 0.45
    if m == 8 and 21 <= d <= 24:                        return 0.60
    if (m == 8 and d >= 25) or (9 <= m <= 11) or (m == 12 and d <= 15): return 1.00
    return 0.80


print('Academic multiplier spot-checks:')
for m, d, label in [(1,1,'New Year/winter break'),(1,20,'Spring semester'),
                    (3,12,'Spring break'),(7,15,'Summer 2'),
                    (8,22,'Orientation'),(9,15,'Fall semester'),(12,20,'Winter break')]:
    val = get_academic_multiplier(pd.Timestamp(year=2023, month=m, day=d))
    print(f'  {label:30s}: {val:.2f}')


Academic multiplier spot-checks:
  New Year/winter break         : 0.15
  Spring semester               : 1.00
  Spring break                  : 0.30
  Summer 2                      : 0.45
  Orientation                   : 0.60
  Fall semester                 : 1.00
  Winter break                  : 0.15


In [ ]:
def get_day_context(date, event_days, holiday_set):
    """
    Single entry point for all date-level context.

    Returns a dict:
      day_type     : 'holiday' | 'event' | 'weekend' | 'weekday'
      waste_mult   : multiplier applied to every generated waste weight
      traffic_mult : multiplier for foot-traffic and event-count lambda
      is_event     : bool
      is_holiday   : bool

    Priority: holiday > event > weekend > weekday.
    The academic calendar is folded into both multipliers.
    """
    date_norm  = pd.Timestamp(date).normalize()
    is_holiday = date_norm in holiday_set
    is_event   = date_norm in event_days
    is_weekend = date_norm.weekday() >= 5
    acad       = get_academic_multiplier(date_norm)

    if is_holiday:
        # Near-closure on public holidays — a handful of records may still appear
        return {'day_type': 'holiday',
                'waste_mult':   0.15 * acad, 'traffic_mult': 0.10 * acad,
                'is_event': False, 'is_holiday': True}

    if is_event:
        return {'day_type': 'event',
                'waste_mult':   1.45 * acad, 'traffic_mult': 1.80 * acad,
                'is_event': True, 'is_holiday': False}

    if is_weekend:
        return {'day_type': 'weekend',
                'waste_mult':   0.65 * acad, 'traffic_mult': 0.60 * acad,
                'is_event': False, 'is_holiday': False}

    return {'day_type': 'weekday',
            'waste_mult':   1.00 * acad, 'traffic_mult': 1.00 * acad,
            'is_event': False, 'is_holiday': False}


# Quick verification
print('Day context spot-checks (2023):')
_test_events    = {pd.Timestamp('2023-09-15')}
_test_holidays  = build_holiday_set(2023, 2023)
for ts_str in ['2023-01-01','2023-09-15','2023-11-11','2023-10-07','2023-07-15']:
    ctx = get_day_context(pd.Timestamp(ts_str), _test_events, _test_holidays)
    print(f"  {ts_str}  type={ctx['day_type']:8s}  "
          f"waste_mult={ctx['waste_mult']:.3f}  traffic_mult={ctx['traffic_mult']:.3f}")


Day context spot-checks (2023):
  2023-01-01  type=holiday   waste_mult=0.022  traffic_mult=0.015
  2023-09-15  type=event     waste_mult=1.450  traffic_mult=1.800
  2023-11-11  type=holiday   waste_mult=0.150  traffic_mult=0.100
  2023-10-07  type=weekend   waste_mult=0.650  traffic_mult=0.600
  2023-07-15  type=weekend   waste_mult=0.293  traffic_mult=0.270


In [ ]:
def generate_event_count(meal, day_of_week, traffic_mult, baseline_events):
    """
    Poisson-sample the number of waste-event records for a given meal on a given day.
    Lambda = baseline * traffic_mult.
    """
    base = baseline_events.loc[(meal, day_of_week)]
    lam  = max(base * traffic_mult, 0.5)
    return np.random.poisson(lam)


def generate_event_timestamp(base_date, meal):
    """
    Sample a 30-min-aligned timestamp using slot probabilities proportional to
    the Gaussian intra-meal peak curve.  Peak slots are drawn more often.
    """
    h, m = weighted_slot_sample(meal)
    return pd.Timestamp(base_date) + pd.Timedelta(hours=h, minutes=m)


def generate_waste_weight(category, waste_multiplier=1.0):
    """
    Generate a waste weight for one event.

    Flow
    ----
    1. Draw from the category-specific log-normal.
    2. Scale by waste_multiplier (combines day type, academic phase, intra-meal).
    3. Cap at 5 kg (original data max) to stay in-distribution.

    The multiplier is the single lever for all external influences — no
    compounding of separate meal/section factors that skewed the original.
    """
    params    = category_params[category]
    log_waste = np.random.normal(params['mean_log'], params['std_log'])
    waste     = np.exp(log_waste) * waste_multiplier
    return max(min(round(waste, 2), 5.0), 0.01)


def generate_background_waste():
    """
    Generate a near-zero background waste value for an uncovered 30-min slot.

    These slots represent minimal-activity periods (cleaning, restocking, or
    a stray early/late visitor).  Exponential distribution gives the correct
    right-skewed low-value shape.

    Mean ≈ 0.05 kg  |  95th pct ≈ 0.15 kg  |  hard cap at 0.30 kg.

    This is intentionally far below the event distribution (median ≈ 2 kg)
    so any time-series model can cleanly distinguish peak from off-peak.
    """
    waste = np.random.exponential(scale=0.05)
    return max(min(round(waste, 3), 0.30), 0.005)


def generate_derived_features(timestamp, waste_weight, unit_price, foot_traffic):
    """Compute all date-time derived columns."""
    cost_loss       = waste_weight * unit_price
    waste_per_price = waste_weight / unit_price if unit_price > 0 else 0
    log_waste       = np.log(max(waste_weight, 1e-6))
    wday_num        = timestamp.weekday()
    is_weekend      = 1 if wday_num >= 5 else 0
    m               = timestamp.month
    return {
        'Year':         timestamp.year,   'Month':     m,
        'Day':          timestamp.day,    'Hour':      timestamp.hour,
        'Minute':       timestamp.minute,
        'Weekday':      timestamp.strftime('%A'),
        'Week':         timestamp.isocalendar()[1],
        'DayOfYear':    timestamp.timetuple().tm_yday,
        'Quarter':      (m - 1) // 3 + 1,
        'IsWeekend':    is_weekend,
        'Month_Name':   timestamp.strftime('%B'),
        'Weekday_Type': 'Weekend' if is_weekend else 'Weekday',
        'Cost_Loss':        round(cost_loss, 2),
        'Waste_per_Price':  round(waste_per_price, 6),
        'Log_Waste':        round(log_waste, 6),
        'Foot_Traffic':     round(foot_traffic, 2),
    }


def apply_flag_thresholds(df, thresholds):
    """Add boolean flag columns based on global IQR thresholds."""
    df['Is_Waste_Outlier'] = ((df['Waste_Weight_kg'] < thresholds['waste_lower']) |
                              (df['Waste_Weight_kg'] > thresholds['waste_upper'])).astype(int)
    df['Is_Cost_Outlier']  = ((df['Cost_Loss'] < thresholds['cost_lower']) |
                              (df['Cost_Loss'] > thresholds['cost_upper'])).astype(int)
    df['Is_High_Waste']    = (df['Waste_Weight_kg'] > thresholds['high_waste']).astype(int)
    df['Is_High_Cost']     = (df['Cost_Loss'] > thresholds['high_cost']).astype(int)
    return df

print('All generation functions defined.')


All generation functions defined.


## 6. Define Special Days
We build the full US public holiday set for 2015-2025 and randomly assign 100 campus event days (events are never placed on public holidays).


In [ ]:
target_start = datetime(2015, 1, 1)
original_max_date = df_original['Date'].max()
target_end = max(original_max_date, pd.Timestamp('2024-12-31')).to_pydatetime()

holiday_set = build_holiday_set(target_start.year, target_end.year + 1)
print(f'Public holidays in period: {len(holiday_set)}')
print('Sample dates:', sorted(holiday_set)[:6])


def create_event_days(start_date, end_date, holiday_set, n_events=100, seed=42):
    """
    Randomly select n_events unique days, excluding public holidays.
    Events are not scheduled on public holidays to keep the two signals clean.
    """
    np.random.seed(seed)
    date_range = pd.date_range(start_date, end_date, freq='D')
    eligible   = [d for d in date_range if d.normalize() not in holiday_set]
    chosen     = np.random.choice(eligible, size=n_events, replace=False)
    return set(pd.Timestamp(d).normalize() for d in chosen)


event_days = create_event_days(target_start, target_end, holiday_set, n_events=100, seed=42)
print(f'\nSynthetic period: {target_start.date()} to {target_end.date()}')
print(f'Campus event days selected: {len(event_days)}')
print('First 5:', sorted(event_days)[:5])


Public holidays in period: 144
Sample dates: [Timestamp('2015-01-01 00:00:00'), Timestamp('2015-01-19 00:00:00'), Timestamp('2015-02-16 00:00:00'), Timestamp('2015-05-25 00:00:00'), Timestamp('2015-07-04 00:00:00'), Timestamp('2015-09-07 00:00:00')]

Synthetic period: 2015-01-01 to 2025-08-10
Campus event days selected: 100
First 5: [Timestamp('2015-01-16 00:00:00'), Timestamp('2015-01-20 00:00:00'), Timestamp('2015-02-04 00:00:00'), Timestamp('2015-04-07 00:00:00'), Timestamp('2015-05-04 00:00:00')]


## 7. Generate Synthetic Records

For each day × meal:
1. Compute `get_day_context()` → `waste_mult`, `traffic_mult`, flags.
2. For each canteen section, Poisson-sample event count (lambda scaled by `traffic_mult × section_meal_pref`).
3. Assign each event a slot drawn from the weighted Gaussian peak distribution.
4. Generate waste weight = log-normal × `waste_mult` × `intra_meal_mult`.
5. Fill uncovered slots with `Exponential(0.05 kg)` background readings.


In [ ]:
synthetic_rows = []
date_range_gen = pd.date_range(target_start, target_end, freq='D')

for date in date_range_gen:
    ctx      = get_day_context(date, event_days, holiday_set)
    day_type = ctx['day_type']

    for meal in meal_dist.index:
        covered_slots = set()
        cat_probs     = category_given_meal.loc[meal]

        # ── Event-based records ──────────────────────────────────────
        for section in section_dist.index:
            sec_meal_mult  = SECTION_MEAL_PREF[section][meal]
            effective_traf = ctx['traffic_mult'] * sec_meal_mult
            n_events       = generate_event_count(meal, date.weekday(),
                                                  effective_traf, baseline_events)

            for _ in range(n_events):
                category  = np.random.choice(cat_probs.index, p=cat_probs.values)

                # Slot weighted by intra-meal Gaussian peak
                timestamp = generate_event_timestamp(date, meal)
                slot_key  = (timestamp.hour, timestamp.minute)
                covered_slots.add(slot_key)

                # Waste weight: log-normal × day-type × academic × intra-meal
                intra_mult   = get_intra_meal_multiplier(timestamp.hour, timestamp.minute, meal)
                waste_mult   = ctx['waste_mult'] * intra_mult
                waste_weight = generate_waste_weight(category, waste_multiplier=waste_mult)
                unit_price   = price_map[category]

                foot_traffic = (base_traffic_per_section[section]
                                * intra_mult * ctx['traffic_mult'])

                row = {
                    'Date':             timestamp,
                    'Meal':             meal,
                    'Canteen_Section':  section,
                    'Food_Category':    category,
                    'Waste_Weight_kg':  waste_weight,
                    'Unit_Price_per_kg': unit_price,
                    'Is_Event_Day':     int(ctx['is_event']),
                    'Is_Holiday':       int(ctx['is_holiday']),
                    'Day_Type':         day_type,
                }
                row.update(generate_derived_features(
                    timestamp, waste_weight, unit_price, foot_traffic))
                synthetic_rows.append(row)

        # ── Background fill: near-zero for uncovered slots ───────────
        for (h, m) in meal_30min_slots(meal):
            if (h, m) in covered_slots:
                continue

            timestamp    = date + timedelta(hours=h, minutes=m)
            section      = np.random.choice(section_dist.index, p=section_dist.values)
            category     = np.random.choice(cat_probs.index, p=cat_probs.values)
            waste_weight = generate_background_waste()   # Exponential ~0.05 kg
            unit_price   = price_map[category]
            foot_traffic = base_traffic_per_section[section] * 0.05 * ctx['traffic_mult']

            row = {
                'Date':             timestamp,
                'Meal':             meal,
                'Canteen_Section':  section,
                'Food_Category':    category,
                'Waste_Weight_kg':  waste_weight,
                'Unit_Price_per_kg': unit_price,
                'Is_Event_Day':     int(ctx['is_event']),
                'Is_Holiday':       int(ctx['is_holiday']),
                'Day_Type':         day_type,
            }
            row.update(generate_derived_features(
                timestamp, waste_weight, unit_price, foot_traffic))
            synthetic_rows.append(row)

df_synthetic = pd.DataFrame(synthetic_rows)
print(f'Synthetic data shape: {df_synthetic.shape}')
print('\nSynthetic Waste_Weight_kg stats:')
print(df_synthetic['Waste_Weight_kg'].describe())
print('\nDay type breakdown (record count):')
print(df_synthetic.groupby('Day_Type').size())


Synthetic data shape: (119616, 25)

Synthetic Waste_Weight_kg stats:
count    119616.000000
mean          1.319383
std           1.381625
min           0.005000
25%           0.240000
50%           0.870000
75%           1.890000
max           5.000000
Name: Waste_Weight_kg, dtype: float64

Day type breakdown (record count):
Day_Type
event       5135
holiday     2385
weekday    85725
weekend    26371
dtype: int64


## 7b. Validate Temporal Realism and Scale Alignment
Four checks:
1. Peak slots have higher mean waste than off-peak slots within the same meal.
2. Overall event-record scale matches the original data.
3. Day-type hierarchy is visible: event > weekday > weekend > holiday.
4. Background fill is clearly lower than event records.


In [ ]:
event_mask = df_synthetic['Waste_Weight_kg'] > 0.30
df_events  = df_synthetic[event_mask].copy()

# (1) Intra-meal temporal pattern
print('=== (1) Mean waste by hour within each meal (event records) ===')
for meal, grp in df_events.groupby('Meal'):
    hourly = grp.groupby('Hour')['Waste_Weight_kg'].mean().round(3)
    print(f'  {meal}: {hourly.to_dict()}')

# (2) Scale check
print('\n=== (2) Scale validation ===')
print(f"{'':<15} {'Original':>12} {'Synthetic':>12}")
print('-' * 42)
for stat in ['mean','median','std','min','max']:
    o = getattr(df_original['Waste_Weight_kg'], stat)()
    s = getattr(df_events['Waste_Weight_kg'],   stat)()
    flag = '  <- CHECK' if abs(o - s) / max(o, 1e-9) > 0.5 else ''
    print(f"{stat:<15} {o:>12.3f} {s:>12.3f}{flag}")

# (3) Day-type hierarchy
print('\n=== (3) Mean waste by day type (event records) ===')
print(df_events.groupby('Day_Type')['Waste_Weight_kg']
      .agg(['mean','median','count']).round(3))

# (4) Background fill sanity check
bg_mask = df_synthetic['Waste_Weight_kg'] <= 0.30
print(f"\n=== (4) Background fill ===")
print(f"Background records: {bg_mask.sum()} ({bg_mask.mean()*100:.1f}% of all rows)")
print(df_synthetic[bg_mask]['Waste_Weight_kg'].describe().round(4))


=== (1) Mean waste by hour within each meal (event records) ===
  Breakfast: {6: 1.04, 7: 2.026, 8: 1.634}
  Dinner: {17: 1.786, 18: 2.065, 19: 1.373}
  Lunch: {11: 1.707, 12: 2.043, 13: 1.23}

=== (2) Scale validation ===
                    Original    Synthetic
------------------------------------------
mean                   2.586        1.792
median                 2.600        1.330
std                    1.408        1.355
min                    0.100        0.310  <- CHECK
max                    5.000        5.000

=== (3) Mean waste by day type (event records) ===
           mean  median  count
Day_Type                      
event     2.328    1.92   4644
holiday   0.649    0.50    158
weekday   1.852    1.39  66331
weekend   1.384    1.01  15261

=== (4) Background fill ===
Background records: 33222 (27.8% of all rows)
count    33222.0000
mean         0.0894
std          0.0855
min          0.0050
25%          0.0210
50%          0.0560
75%          0.1400
max          0.3000

## 8. Add Flag Columns


In [ ]:
df_synthetic = apply_flag_thresholds(df_synthetic, thresholds)
print('Flag columns added.')
print(df_synthetic[['Is_Waste_Outlier','Is_Cost_Outlier',
                     'Is_High_Waste','Is_High_Cost']].sum())


Flag columns added.
Is_Waste_Outlier       0
Is_Cost_Outlier     4120
Is_High_Waste       7293
Is_High_Cost        3657
dtype: int64


## 9. Save Augmented Dataset
The final CSV includes two new columns vs the original: `Is_Holiday` and `Day_Type`.


In [ ]:
output_filename = 'data/food_waste_augmented_cl_updated.csv'
df_synthetic.to_csv(output_filename, index=False)
print(f'File saved as {output_filename}')
print(f'Shape: {df_synthetic.shape}')
print(f'Columns: {list(df_synthetic.columns)}')


File saved as data/food_waste_augmented_cl_updated.csv
Shape: (119616, 29)
Columns: ['Date', 'Meal', 'Canteen_Section', 'Food_Category', 'Waste_Weight_kg', 'Unit_Price_per_kg', 'Is_Event_Day', 'Is_Holiday', 'Day_Type', 'Year', 'Month', 'Day', 'Hour', 'Minute', 'Weekday', 'Week', 'DayOfYear', 'Quarter', 'IsWeekend', 'Month_Name', 'Weekday_Type', 'Cost_Loss', 'Waste_per_Price', 'Log_Waste', 'Foot_Traffic', 'Is_Waste_Outlier', 'Is_Cost_Outlier', 'Is_High_Waste', 'Is_High_Cost']


## 10. Quick Verification of Temporal Coverage
Confirm:
1. All 30-min meal-hour windows are populated.
2. Holiday and summer-break rows are distinguishable in volume.
3. Section-level record counts reflect specialisation.


In [ ]:
df_synthetic = df_synthetic.sort_values('Date')
agg = df_synthetic.set_index('Date').resample('30min')['Waste_Weight_kg'].sum()

meal_hours = set()
for start, end in meal_time_windows.values():
    for h in range(start, end):
        meal_hours.add(h)

agg_meal        = agg[agg.index.hour.isin(meal_hours)]
total_windows   = len(agg_meal)
zero_windows    = (agg_meal == 0).sum()
nonzero_windows = (agg_meal > 0).sum()

print(f'Total 30-min meal-hour windows: {total_windows}')
print(f'Zero windows (meal hours only):  {zero_windows} ({zero_windows/total_windows*100:.1f}%)')
print(f'Non-zero windows:               {nonzero_windows} ({nonzero_windows/total_windows*100:.1f}%)')
print(f"\nDate range: {df_synthetic['Date'].min()} to {df_synthetic['Date'].max()}")

event_mask = df_synthetic['Waste_Weight_kg'] > 0.30
print('\n=== Final scale check ===')
for label, ser in [('Original',          df_original['Waste_Weight_kg']),
                   ('Synthetic (events)', df_synthetic[event_mask]['Waste_Weight_kg'])]:
    print(f"{label:22s}: mean={ser.mean():.3f}  median={ser.median():.3f}  "
          f"std={ser.std():.3f}  max={ser.max():.2f}")

print('\nMean event-waste by day type:')
print(df_synthetic[event_mask].groupby('Day_Type')['Waste_Weight_kg'].mean().round(3))

print('\nRecord counts by canteen section:')
print(df_synthetic.groupby('Canteen_Section').size())

print('\nMean event-waste by section and meal:')
print(df_synthetic[event_mask].groupby(['Canteen_Section','Meal'])['Waste_Weight_kg']
      .mean().round(3).unstack())


Total 30-min meal-hour windows: 69750
Zero windows (meal hours only):  0 (0.0%)
Non-zero windows:               69750 (100.0%)

Date range: 2015-01-01 06:00:00 to 2025-08-10 19:30:00

=== Final scale check ===
Original              : mean=2.586  median=2.600  std=1.408  max=5.00
Synthetic (events)    : mean=1.792  median=1.330  std=1.355  max=5.00

Mean event-waste by day type:
Day_Type
event      2.328
holiday    0.649
weekday    1.852
weekend    1.384
Name: Waste_Weight_kg, dtype: float64

Record counts by canteen section:
Canteen_Section
A    29664
B    30719
C    29207
D    30026
dtype: int64

Mean event-waste by section and meal:
Meal             Breakfast  Dinner  Lunch
Canteen_Section                          
A                    1.750   1.797  1.786
B                    1.764   1.835  1.803
C                    1.731   1.844  1.769
D                    1.778   1.844  1.798
